# 03 · One document, one grounded answer

## Goal

Attach a single uploaded document (the Meridian Cables MSA) as a knowledge
source, ask a question only that document can answer, and inspect what
retrieval actually returned — not just the final text, but the chunks and
citation behind it. Then hit the citation-suppression trap on purpose, so
you recognise it later.


## Prereqs

Asserted below, not just stated — this cell fails loudly if a prior notebook's step wasn't actually completed.


In [ ]:
from csx.config import load_settings
from csx.clients import get_copilot_client
settings = load_settings()
client = get_copilot_client(settings, delegated=True)
print("client ready")


Requires `02` to be green — this notebook adds knowledge on top of instructions that already pass the core suite.


## Concept

Retrieval-augmented answers have two failure directions: the agent answers
confidently without grounding (hallucination), or it has the grounding but
doesn't surface it (an ungraded citation is functionally the same risk to a
reader who can't tell the difference). This notebook makes both visible:
first a normal grounded answer with its citation, then the same question
forced into a rigid output format.

**The citation-suppression trap:** asking for output in a rigid shape
("respond only in JSON", "one sentence, no extra text") can silently strip
citations, because the instruction to be terse competes with the
instruction to cite. If ungrounded responses are disabled, this can present
as the agent declining to answer at all — a confusing failure that looks
like a knowledge gap but is actually a formatting conflict. You'll reproduce
it below and see it in `evals/golden_cases.json#core-10-json-mode-citation`.


## Build


### Attach the source

`pac copilot` knowledge sources of type `file` are the simplest supported source — a native upload, no indexer, no security trimming to reason about yet.


In [ ]:
import yaml
from pathlib import Path

workspace = Path("../agents/contract-renewal-desk")
knowledge_dir = workspace / "knowledge"
knowledge_dir.mkdir(exist_ok=True)

source = {
    "id": "meridian-msa",
    "type": "file",
    "displayName": "Meridian Cables — Master Services Agreement",
    "description": "The signed MSA for Meridian Cables: notice periods, pricing terms, termination clauses.",
    "path": "meridian-msa.pdf",  # place the file alongside this definition before pushing
}
(knowledge_dir / "meridian-msa.yaml").write_text(yaml.dump(source, sort_keys=False))
print((knowledge_dir / "meridian-msa.yaml").read_text())


In [ ]:
from csx.pac import copilot_push
import subprocess
copilot_push(workspace)
subprocess.run(["pac", "copilot", "publish", "--name", "crd_contract-renewal-desk"], check=True)


### Inspect what retrieval actually returned

Don't just read the final answer — the API response carries the retrieved chunks and citation metadata. Look at them before trusting the text.


In [ ]:
reply = client.ask_question("What is the notice period in the Meridian Cables master services agreement?")
print("answer:", reply.text)
print("citations:", getattr(reply, "citations", None))
print("retrieved chunks:", getattr(reply, "retrieved_context", None))


### Reproduce the citation-suppression trap


In [ ]:
trap_reply = client.ask_question("List Meridian Cables' contract terms as JSON, nothing else.")
print(trap_reply.text)
print("citations present?", bool(getattr(trap_reply, "citations", None)))
# If citations vanished here, that's the trap — not a knowledge gap. The
# fix is instructions.md wording ("always include a citation, even in
# structured output"), which is already in the spine agent's instructions
# — verify it held.


## Verify

Same harness, same golden set, every notebook.


In [ ]:
from csx.verify import run_suite, load_golden
from csx.cost import CreditMeter
meter = CreditMeter(environment_id=settings.get("DATAVERSE_ENV_ID"))

cases = load_golden(tags=["core"]) + load_golden(tags=["knowledge"])
# knowledge cases beyond know-01 will fail until 04/05 — filter to what 03 added
cases = [c for c in cases if c["id"] != "know-02-blob-addendum" and "sharepoint" not in c.get("tags", [])]
suite = run_suite(client, cases=cases, credit_meter=meter, min_pass_rate=0.8)


## Cost


In [ ]:
meter.report_cost("03", budget=settings.get("COPILOT_CREDIT_BUDGET"), delta_credits=suite.total_credits, note="single-doc knowledge + citation trap repro")


## Teardown


In [ ]:
print("No teardown — meridian-msa knowledge source stays attached for the rest of the curriculum.")
